In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [14]:
MODEL_NAME = "DeepMount00/Mistral-RAG"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).eval()
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████| 3/3 [00:03<00:00,  1.30s/it]


In [15]:
def generate_answer(prompt, response_type="generativo"):
    # Creazione del contesto e della domanda in base al tipo di risposta
    if response_type == "estrattivo":
        prompt = f"Rispondi alla seguente domanda in modo estrattivo, basandoti esclusivamente sul contesto.\n{prompt}"
    else:
        prompt = f"Rispondi alla seguente domanda in modo generativo, basandoti esclusivamente sul contesto.\n{prompt}"

    # Preparazione del messaggio per il modello
    messages = [
        {"role": "user", "content": prompt},
    ]
    model_inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(device)
    generated_ids = model.generate(model_inputs, max_new_tokens=250, do_sample=True,
                                   temperature=0.001, eos_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    return decoded[0].split("[/INST]", 1)[1].strip() if "[/INST]" in decoded[0] else "Errore nella generazione della risposta"


In [6]:
contesto = """Venerdì più di 2.100 persone che vivono vicino a un vulcano in Indonesia sono state sfollate per i rischi legati a un’eruzione. Martedì infatti l’isola vulcanica di Ruang, che si trova circa 100 chilometri a nord di Sulawesi, ha cominciato a eruttare, producendo una colonna di fumo e ceneri che ieri ha raggiunto 1.200 metri di altezza. Le operazioni di evacuazione sono ancora in corso: complessivamente sono più di 11mila le persone a cui è stato detto di lasciare le proprie case. Gran parte di loro vive sulla vicina isola di Tagulandang, che in totale ha 20mila abitanti; potrebbe essere raggiunta non solo dalle ceneri vulcaniche e dai piroclasti, ma anche da un eventuale tsunami causato dalla caduta in mare di lava e rocce."""
domanda = "Perchè le persone sono evacuate dalle case?"
prompt = f"Contesto: {contesto}\nDomanda: {domanda}"

In [10]:
contesto = """
Ora        ProduzioneSolare(kW)      ConsumiTotali(kW)          StatoBatteria(%)     EnergiaAcquistata(kW)        AccensioneTLC
2025-03-27 15:00:00      6.28   0.10   100  0.0  NO
2025-03-27 16:00:00      3.1   4.10   90   0.0   Sì
2025-03-27 17:00:00      1.21   0.10   100  0.0   NO
2025-03-27 18:00:00      0.0   0.10   99   0.0   NO
2025-03-27 19:00:00      0.0   0.10   98   0.0   NO
2025-03-27 20:00:00      0.0   4.10   73   1.6   Sì
2025-03-27 21:00:00      0.0   4.10   48   1.6   Sì
2025-03-27 22:00:00      0.0   4.10   23   1.6   Sì
2025-03-27 23:00:00      0.0   4.10   10   2.8   Sì
2025-03-28 00:00:00      0.0   0.10   10   0.1   NO
2025-03-28 01:00:00      0.0   0.10   10   0.1   NO
2025-03-28 02:00:00      0.0   0.10   10   0.1   NO
2025-03-28 03:00:00      0.0   0.10   10   0.1   NO"""
domanda = "quando accendere le pompe di calore"
prompt = f"Contesto: {contesto}\nDomanda: {domanda}"

In [7]:
contesto = """
PROSUMER 1
Ora        ProduzioneSolare(kW)      ConsumiTotali(kW)
2025-03-27 15:00:00      6.28   0.10
2025-03-27 16:00:00      3.1   4.10
2025-03-27 17:00:00      1.21   0.10
2025-03-27 18:00:00      0.0   0.10
2025-03-27 19:00:00      0.0   0.10 
2025-03-27 20:00:00      0.0   4.10 

PROSUMER 2
Ora        ProduzioneSolare(kW)      ConsumiTotali(kW) 
2025-03-27 15:00:00      0.0   0.10 
2025-03-27 16:00:00      0.0   4.10  
2025-03-27 17:00:00      1.21   0.10 
2025-03-27 18:00:00      0.0   0.10 
2025-03-27 19:00:00      0.0   0.10 
2025-03-27 20:00:00      0.0   4.10 

PROSUMER 3
Ora        ProduzioneSolare(kW)      ConsumiTotali(kW)
2025-03-27 15:00:00      3.0   4.10 
2025-03-27 16:00:00      2.0   3.10 
2025-03-27 17:00:00      1.21   0.10 
2025-03-27 18:00:00      0.0   0.10 
2025-03-27 19:00:00      0.0   3.10 
2025-03-27 20:00:00      0.0   4.10 

PROSUMER 4
Ora        ProduzioneSolare(kW)      ConsumiTotali(kW) 
2025-03-27 15:00:00      0.0   0.10
2025-03-27 16:00:00      0.0   4.10
2025-03-27 17:00:00      1.21   0.10 
2025-03-27 18:00:00      0.0   0.10 
2025-03-27 19:00:00      0.0   0.10
2025-03-27 20:00:00      0.0   4.10

PROSUMER 5
Ora        ProduzioneSolare(kW)      ConsumiTotali(kW)
2025-03-27 15:00:00      0.0   0.10
2025-03-27 16:00:00      0.0   4.10
2025-03-27 17:00:00      1.21   0.10
2025-03-27 18:00:00      0.0   0.10
2025-03-27 19:00:00      0.0   0.10
2025-03-27 20:00:00      0.0   4.10

"""
domanda = "quando conviene al PROSUMER 2 vendere la propria energia solare in eccesso?"
prompt = f"Contesto: {contesto}\nDomanda: {domanda}"

In [16]:
prompt = """
Stai supportando un prosumer appartenente a una comunità energetica composta da membri che possono essere o prosumer (con produzione e consumo giornalieri di energia) o consumer (solo consumo, nessuna produzione).

La comunità scambia energia utilizzando una blockchain:
- Ogni kWh è rappresentato da un NFT ERC721.
- Ogni centesimo di euro è rappresentato da 1 token ERC20.
- I membri scelgono liberamente a che prezzo vendere l’energia in surplus, con un vincolo:
  - Il prezzo deve essere maggiore di 0,04 € (prezzo di vendita alla rete)
  - E minore di 0,16 € (prezzo di acquisto dalla rete)

Applica le seguenti regole per stabilire a chi può essere venduto il surplus:

1. Ordine di priorità: 
   - Prima i prosumer in deficit (cioè coloro che hanno consumato più di quanto prodotto).
   - Solo il surplus residuo può essere venduto ai consumer.
2. Ogni transazione è espressa in NFT (1 kWh) + corrispondente quantità di token (in cent).
3. L’obiettivo è vendere quanto più possibile del proprio surplus, ottimizzando il prezzo.

Di seguito trovi i dati giornalieri dei membri della comunità (in kWh):


- Peer1 (utente): produzione = 20.73, consumo = 10.03

- Peer2: produzione = 20.04, consumo = 11.04

- Peer3: produzione = 9.63, consumo = 3.28

- Peer4: consumo = 0.45

- Peer5: consumo = 5.59


Sulla base dei dati sopra, fornisci un suggerimento **per il prosumer utente (che in questo caso è Peer1)** indicando:
1. A chi può vendere il surplus, in quale ordine e in che misura.
2. A quale prezzo (in centesimi di euro) vendere ciascun blocco di kWh, tenendo conto delle regole.

Riassumi il suggerimento in modo chiaro ed efficace (es. tabella o elenco).
"""

In [ ]:
import time

start = time.time()
response = generate_answer(prompt, "estrattivo")
tot = time.time() - start
print(response)

In [19]:
import os, csv
from datetime import datetime
LOG_FILE = "results.csv"

def save_log(model, prompt, response, gen_time):
    file_exists = os.path.isfile(LOG_FILE)
    with open(LOG_FILE, 'a', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["timestamp", "modello", "prompt", "risposta", "tempi esecuzione"])
        writer.writerow([datetime.now().isoformat(), model, prompt, response, gen_time])

model_name = "MistralRAG_estrattivo"
save_log(model_name, prompt, response, tot)

In [20]:
start = time.time()
response = generate_answer(prompt, "generativo")
tot = time.time() - start
print(response)
model_name = "MistralRAG_generativo"
save_log(model_name, prompt, response, tot)

Risposta: Per il prosumer utente Peer1, che ha una produzione di 20.73 kWh e un consumo di 10.03 kWh, può vendere il surplus di 10.7 kWh (20.73 - 10.03) seguendo le regole:

1. Ordine di priorità:
   - Prima vendere al Peer2, che ha un consumo in eccesso di 1.8 kWh (20.04 - 11.04).
   - Se il surplus non è sufficiente per soddisfare il deficit di Peer2, vendere al Peer3, che ha un consumo in eccesso di 6.35 kWh (9.63 - 3.28).
   - Se il surplus non è sufficiente per soddisfare il deficit di Peer3, vendere al Peer5, che ha un consumo di 5.59 kWh.

2. Prezzo di vendita:
   - Per ottimizz
